# Lab 03: LangChain Integration

**Goal:** Use CallbackHandler to trace LangChain chains and agents with LangFuse.

**What you'll learn:**
- How to integrate LangFuse with LangChain using CallbackHandler
- How to trace RAG pipelines end-to-end
- Where trace attributes live in langfuse v4 (run config, not handler kwargs)
- How to instrument chains with a single line of code

In [ ]:
import os
import shutil
import textwrap

WORKDIR = "/tmp/k8s-lab-12-03"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

## Step 1: CallbackHandler Pattern

In [ ]:
print("One-line integration with LangChain:\n")

code_example = textwrap.dedent("""\
    from langfuse import Langfuse
    from langfuse.langchain import CallbackHandler

    langfuse = Langfuse()   # reads LANGFUSE_PUBLIC_KEY / SECRET_KEY / HOST

    # v4: the handler itself carries no trace metadata.
    # Create the trace id up front so you can attach scores to it later.
    trace_id = langfuse.create_trace_id()
    handler = CallbackHandler(trace_context={\"trace_id\": trace_id})

    # Trace attributes travel in the run config, not the handler
    result = chain.invoke(
        {\"question\": \"What is RAG?\"},
        config={
            \"callbacks\": [handler],
            \"run_name\": \"rag_query\",
            \"metadata\": {
                \"langfuse_user_id\": \"alice\",
                \"langfuse_session_id\": \"chat_42\",
                \"langfuse_tags\": [\"production\", \"v2.0\"],
            },
        },
    )
""")

for line in code_example.strip().split("\n"):
    print(f"    {line}")

In [ ]:
print("What gets captured automatically:")
captured = [
    ("LLM calls",      "Model, input, output, tokens, cost, latency"),
    ("Tool calls",      "Tool name, input, output, duration"),
    ("Chain execution", "Each chain step with input/output"),
    ("Retriever calls", "Query, retrieved documents, scores"),
]
for what, detail in captured:
    print(f"    {what:<18} {detail}")

## Step 2: RAG Pipeline Tracing

In [ ]:
rag_example = textwrap.dedent("""\
    from langchain.chains import RetrievalQA
    from langchain_community.vectorstores import Chroma

    # Set up retriever
    vectorstore = Chroma(persist_directory="./chroma_db")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    # Create chain
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
    )

    # Invoke with LangFuse callback
    result = chain.invoke(
        {"query": "How does RAG work?"},
        config={"callbacks": [handler]},
    )
""")

print("RAG pipeline with LangFuse tracing:\n")
for line in rag_example.strip().split("\n"):
    print(f"    {line}")

In [ ]:
print("Trace structure for RAG:")
print("    Trace: chat_request")
print("    \u251c\u2500\u2500 Span: retriever")
print("    \u2502   \u2514\u2500\u2500 (query: 'How does RAG work?', docs: 5)")
print("    \u2514\u2500\u2500 Span: llm_chain")
print("        \u2514\u2500\u2500 Generation: groq/llama3-70b")
print("            \u251c\u2500\u2500 input_tokens: 1,200")
print("            \u251c\u2500\u2500 output_tokens: 350")
print("            \u2514\u2500\u2500 cost: $0.0062")

## Step 3: Handler Configuration

In [ ]:
print("Where trace attributes go in langfuse v4:\n")
params = [
    ("langfuse_user_id",     "config metadata",    "Identifies the user (cost-per-user tracking)"),
    ("langfuse_session_id",  "config metadata",    "Groups related requests into a conversation"),
    ("langfuse_tags",        "config metadata",    '["production", "v2.0", "experiment"]'),
    ("run_name",             "config key",         'Trace name: "rag_query" or "agent_run"'),
    ("trace_context",        "CallbackHandler()",  "Binds the handler to a known trace_id"),
]
print(f"    {'Attribute':<22} {'Set via':<20} {'Purpose'}")
print(f"    {'-'*84}")
for param, where, purpose in params:
    print(f"    {param:<22} {where:<20} {purpose}")

print()
print("    Custom key/value pairs go in the same config metadata dict alongside")
print("    the langfuse_* keys, e.g. {\"endpoint\": \"/chat\", \"model\": \"llama3\"}.")
print()
print("    Alternative: wrap the call in propagate_attributes(user_id=..., session_id=...,")
print("    tags=[...], trace_name=...) to apply the same attributes to everything inside.")

## TODO 1: LangChain Instrumentation Code

Write code that:
- Imports `CallbackHandler` from `langfuse.langchain`
- Creates a trace id with `langfuse.create_trace_id()`
- Creates the handler with `trace_context={"trace_id": trace_id}`
- Creates a RetrievalQA chain with retriever
- Invokes the chain with `callbacks=[handler]`, a `run_name`, and `langfuse_*` metadata

In [ ]:
# TODO: Instrument a LangChain RAG pipeline with LangFuse (v4 API)
# Include: langfuse.langchain import, create_trace_id, handler, chain invocation

todo1_code = textwrap.dedent("""\
    # TODO: Instrument a LangChain RAG pipeline with LangFuse (v4 API)
    # Include: langfuse.langchain import, create_trace_id, handler, chain invocation

""")

with open(os.path.join(WORKDIR, "instrumented_rag.py"), "w") as f:
    f.write(todo1_code)

In [ ]:
checks1 = [
    ("Has langfuse.langchain import", "langfuse.langchain" in todo1_code),
    ("Has CallbackHandler creation",  "CallbackHandler(" in todo1_code),
    ("Has create_trace_id",           "create_trace_id" in todo1_code),
    ("Has trace_context binding",     "trace_context" in todo1_code),
    ("Has langfuse_user_id",          "langfuse_user_id" in todo1_code),
    ("Has langfuse_session_id",       "langfuse_session_id" in todo1_code),
    ("Has langfuse_tags",             "langfuse_tags" in todo1_code),
    ("Has chain invoke",              "invoke" in todo1_code),
    ("Has callbacks config",          "callbacks" in todo1_code),
    ("Has run_name",                  "run_name" in todo1_code),
]

score1 = sum(1 for _, ok in checks1 if ok)
print(f"Validating ({score1}/{len(checks1)}):\n")
for name, ok in checks1:
    print(f"    [{'PASS' if ok else 'FAIL'}] {name}")

## TODO 2: Integration Quiz

Fill in the answers for each question below.

In [ ]:
quiz = [
    {
        "question": "What LangFuse class integrates with LangChain?",
        "answer": "___",
        "correct": "callbackhandler",
    },
    {
        "question": "What config key passes the handler to chain.invoke()?",
        "answer": "___",
        "correct": "callbacks",
    },
    {
        "question": "What config metadata key groups requests into conversations?",
        "answer": "___",
        "correct": "langfuse_session_id",
    },
    {
        "question": "What trace level captures the actual LLM API call?",
        "answer": "___",
        "correct": "generation",
    },
]

# YOUR CODE HERE: Fill in quiz answers
# quiz[0]["answer"] = "CallbackHandler"
# quiz[1]["answer"] = ???
# ...

In [ ]:
score2 = 0
for i, q in enumerate(quiz, 1):
    answer = q["answer"].strip().lower().replace("_", "").replace(" ", "")
    correct = q["correct"].lower().replace("_", "").replace(" ", "")
    is_correct = answer == correct

    if q["answer"] == "___":
        status = "TODO"
    elif is_correct:
        status = "PASS"
        score2 += 1
    else:
        status = "FAIL"
    print(f"    [{status}] Q{i}: {q['question']}")

print(f"\n  Score: {score2}/{len(quiz)}")

## Summary

In [ ]:
print("Key concepts:")
print("    1. CallbackHandler = one-line LangChain integration")
print("    2. Pass handler via config={'callbacks': [handler]}")
print("    3. Automatically captures LLM calls, tools, chains, retrievers")
print("    4. Set langfuse_user_id + langfuse_session_id in config metadata")
print(f"\n  TODO 1: {score1}/{len(checks1)} instrumentation checks")
print(f"  TODO 2: {score2}/{len(quiz)} quiz answers correct")
print(f"\n  Files generated in {WORKDIR}/")

## Key Takeaways

- **CallbackHandler** provides one-line integration between LangChain and LangFuse
- Pass the handler via `config={"callbacks": [handler]}` to any LangChain `.invoke()` call
- LangFuse automatically captures LLM calls, tool calls, chain execution, and retriever calls
- In v4 the handler takes no trace metadata: put `langfuse_user_id` and `langfuse_session_id` in the run config's `metadata` for per-user cost tracking and conversation grouping
- Use `langfuse_tags` plus your own metadata keys to categorize and enrich traces for filtering in the LangFuse dashboard, and `run_name` to name the trace